In [0]:
# Databricks Notebook: L1_Ingesta_Bronce.ipynb
# ------------------------------------------------------------------------------
# Este notebook es responsable de la ingesta de datos brutos (CSV) desde el
# repositorio de origen y su carga inmutable a la Capa Bronce (L1) en formato Delta Lake.
# ------------------------------------------------------------------------------

import pandas as pd
# IMPORTACIÓN CORREGIDA: Se añade 'lit' a las funciones de PySpark
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql import SparkSession

# Inicializar Spark Session (ya disponible en Databricks)
spark = SparkSession.builder.appName("IngestaBronceRFM").getOrCreate()

# 1. Definición de Rutas y Esquema
# ------------------------------------------------------------------------------
# ATENCIÓN: Se usa la ruta proporcionada por el usuario.
# Esta ruta apunta a un Volume de Unity Catalog o a otra ubicación montada.
CSV_BASE_PATH = "dbfs:/Volumes/olist/olist_csv/olist/" # Nueva Ruta Base

# Usaremos un nombre de esquema que puede requerir el prefijo del Catálogo si no es el default.
# Ej: "main.bronze" si usas el catálogo 'main'.
DELTA_BRONZE_SCHEMA = "bronze" 

print(f"Ruta base de CSVs: {CSV_BASE_PATH}")
print(f"Esquema de destino: {DELTA_BRONZE_SCHEMA}")

# --- CORRECCIÓN CLAVE PARA UNITY CATALOG ---
# En UC, para crear un esquema gestionado, NO SE DEBE usar la clausula 'LOCATION'
# a menos que se use 'MANAGED LOCATION' o se esté creando un esquema externo.
# Asumimos que queremos un esquema GESTIONADO simple dentro del catálogo activo.
try:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DELTA_BRONZE_SCHEMA}")
    print(f"Esquema {DELTA_BRONZE_SCHEMA} creado o asegurado correctamente en el Catálogo activo.")
except Exception as e:
    # Si falla porque necesitas el prefijo de Catálogo (ej: 'main.bronze')
    print(f"❌ ERROR al intentar crear el esquema simple. Posiblemente necesite el prefijo del Catálogo. Intenta: CREATE SCHEMA IF NOT EXISTS TU_CATALOGO.{DELTA_BRONZE_SCHEMA}")
    raise e


# 2. Configuración de Tablas y Archivos
# ------------------------------------------------------------------------------
# Lista de tuplas: (nombre_archivo_csv, nombre_tabla_delta)
FILE_MAPPING = [
    ("olist_customers_dataset.csv", "olist_customers"),
    ("olist_orders_dataset.csv", "olist_orders"),
    ("olist_order_payments_dataset.csv", "olist_order_payments"),
    ("olist_order_items_dataset.csv", "olist_order_items"),
    ("olist_products_dataset.csv", "olist_products"),
    ("product_category_name_translation.csv", "product_category_translation"),
    ("olist_geolocation_dataset.csv", "olist_geolocation"), 
    ("olist_sellers_dataset.csv", "olist_sellers"),
    ("olist_order_reviews_dataset.csv", "olist_reviews"),
]

# 3. Función de Ingesta
# ------------------------------------------------------------------------------
def ingest_to_bronze(csv_filename, delta_tablename):
    """
    Carga un archivo CSV desde la ruta base y lo guarda como tabla Delta
    en la capa Bronce con metadatos de ingesta.
    """
    csv_path = f"{CSV_BASE_PATH}{csv_filename}"
    delta_path = f"{DELTA_BRONZE_SCHEMA}.{delta_tablename}" # Ahora UC-compatible

    try:
        # Lectura del CSV usando inferSchema (ajustar si se necesita esquema estricto)
        df_raw = spark.read.csv(
            csv_path,
            header=True,
            inferSchema=True,
            sep=','  # Asumiendo separador por coma
        )

        # 4. Adición de Metadatos de Ingesta (Campos de Auditoría)
        # CORRECCIÓN: Se usa 'lit' directamente, ya que fue importada arriba.
        df_bronze = df_raw.withColumn(
            "ingestion_timestamp", current_timestamp()
        ).withColumn(
            "source_file", lit(csv_filename)
        )

        # Escritura a la Capa Bronce (Delta Lake)
        # El uso de saveAsTable es compatible con Unity Catalog
        df_bronze.write.format("delta").mode("overwrite").saveAsTable(delta_path)

        print(f"✅ Éxito al cargar '{csv_filename}' a la tabla '{delta_path}' ({df_bronze.count()} filas)")

    except Exception as e:
        print(f"❌ ERROR al procesar '{csv_filename}'. Asegúrate de que el archivo existe en la ruta y el esquema es accesible. Causa: {e}")
        raise e

# 5. Ejecución del Proceso de Ingesta
# ------------------------------------------------------------------------------
print("\n--- INICIO DEL PROCESO DE INGESTA A CAPA BRONCE ---")
for csv_file, delta_table in FILE_MAPPING:
    ingest_to_bronze(csv_file, delta_table)

# 6. Verificación de las Tablas
# ------------------------------------------------------------------------------
print("\n--- VERIFICACIÓN DE LAS TABLAS DE BRONCE ---")
# Esto listará las tablas en el catálogo activo
spark.sql(f"SHOW TABLES IN {DELTA_BRONZE_SCHEMA}").show()

print("\n--- MUESTRA DE DATOS DE CUSTOMERS ---")
# Intenta mostrar una muestra, lo que validará que la tabla se escribió correctamente
try:
    spark.table(f"{DELTA_BRONZE_SCHEMA}.olist_customers").limit(5).toPandas()
except Exception as e:
    print(f"No se pudo leer la tabla de muestra. Asegúrate de que el Catálogo y Esquema sean correctos.")
    print(e)

print("\n*** PROCESO L1 (BRONCE) FINALIZADO EXITOSAMENTE ***")